# AICE Associate 변주 문제 v2 - 이탈 여부, 이탈_여부 예측 (분류)

### 이 노트북의 구성요소

아래 다이어그램은 이 노트북 해설에서 실제로 사용된 함수·클래스·모듈을 구간별로 모은 것입니다 (굵게 표시된 이름은 이 노트북에서 특별히 선택된 항목입니다).

In [ ]:
from IPython.display import HTML, display

display(HTML("""
<script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>

<div style="display:flex; flex-direction:column; gap:6px; width:fit-content;">
<pre class="mermaid">
flowchart LR
    subgraph LIB["라이브러리"]
        LIB1["<b>라이브러리</b><br/>numpy<br/>pandas<br/>pyplot<br/>seaborn"]
    end
    subgraph EXP["데이터 탐색"]
        EXP1["<b>데이터 탐색</b><br/>groupby<br/>head<br/>merge<br/>read_csv<br/>read_json"]
    end
    subgraph PREP["전처리"]
        PREP1["<b>전처리</b><br/><b>RobustScaler</b><br/>LabelEncoder<br/>drop<br/>fillna<br/>get_dummies<br/>info"]
    end
    LIB --> EXP --> PREP
    classDef libCls fill:#F0EEE8,stroke:#8A8478,color:#3A362E
    class LIB libCls
    classDef expCls fill:#E9F1F6,stroke:#5B7C99,color:#1E3548
    class EXP expCls
    classDef prepCls fill:#FAF3E7,stroke:#B08D57,color:#5C4720
    class PREP prepCls
</pre>
<pre class="mermaid">
flowchart LR
    subgraph ML["머신러닝"]
        ML1["<b>머신러닝</b><br/><b>XGBClassifier</b><br/><b>roc_auc_score</b><br/>DataFrame<br/>DecisionTreeClassifier<br/>GridSearchCV<br/>sort_values"]
    end
    subgraph UNSUP["비지도학습"]
        UNSUP1["<b>비지도학습</b><br/>(기본 구성)"]
    end
    subgraph DL["딥러닝"]
        DL1["<b>딥러닝</b><br/><b>ModelCheckpoint</b><br/><b>load_model</b><br/>Dense<br/>Dropout<br/>EarlyStopping<br/>Sequential"]
    end
    ML --> UNSUP --> DL
    classDef mlCls fill:#FBEEEA,stroke:#B5654A,color:#5C2A1B
    class ML mlCls
    classDef unsupCls fill:#F3EEF4,stroke:#8B6F8E,color:#402F42
    class UNSUP unsupCls
    classDef dlCls fill:#EBEEF6,stroke:#445577,color:#232C3D
    class DL dlCls
</pre>
</div>

<script>
mermaid.initialize({ startOnLoad: false, securityLevel: "loose" });
mermaid.run();
</script>
"""))

### 시나리오

통신사 '스타링크'는 고객 유지율을 극대화하고 마케팅 자원을 효율적으로 배분하기 위해 이탈 가능성이 높은 고객을 사전에 예측하고자 합니다. 이 모델을 통해 영업팀은 선제적으로 맞춤형 유지 방안을 제시하고, 이탈 위험이 높은 고객에게 특별 할인 또는 서비스 개선 제안을 하여 고객 이탈을 최소화하는 것을 목표로 합니다.

---

**[유의사항]**
- 답안은 각 문항 아래 표시된 `# (N) 여기에 ...` 칸에 작성하세요.
- **정답/해설은 이 노트북 가장 아래 `## 해설` 섹션에 모아뒀습니다.** 먼저 스스로 풀어본 뒤에 확인하세요.
- 이 노트북은 오리지널 창작 문제이며, 실제 AICE 샘플문항 원문을 복제하지 않습니다.
- 데이터 로더: `read_json` / 기본모델: `DecisionTreeClassifier` / 비교모델: `XGBClassifier` / 스케일러: `RobustScaler`

**[데이터 컬럼 설명]**

- Churned : 이탈 여부, 이탈_여부
- MonthlyCharge : 월별 청구 금액 (원)
- ContractType : 계약 유형 (단기/장기/위약금 없음)
- RegionCode : 지역 코드 (A, B, C, D 등 5개 코드값) (dim.csv 와 병합 키)
- TenureMonths : 피처 컬럼
- TotalUsageGB : 피처 컬럼
- ServiceType : 피처 컬럼
- PaymentMethod : 피처 컬럼
- CustomerID : 식별자(모델링에 불필요)
- (병합 후) dim_value : RegionCode별 평균 이탈률

## 0. 데이터 준비

다음 문항을 풀기 전에 아래 코드를 실행하세요 (문제 데이터 2개 테이블을 생성합니다).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


def synthesize_tables(seed=2008, n_rows=600):
    rng = np.random.default_rng(seed)
    n = n_rows

    base = rng.normal(loc=50, scale=15, size=n)
    outlier_idx = rng.choice(n, size=max(3, n // 50), replace=False)
    base[outlier_idx] += rng.choice([1, -1], size=len(outlier_idx)) * rng.uniform(80, 150, size=len(outlier_idx))

    main_df = pd.DataFrame({'MonthlyCharge': base.round(2)})

    cats = [f"Cat{i+1}" for i in range(rng.integers(3, 5))]
    main_df['ContractType'] = rng.choice(cats, size=n)

    n_regions = rng.integers(5, 9)
    regions = [f"R{i+1:02d}" for i in range(n_regions)]
    main_df['RegionCode'] = rng.choice(regions, size=n)

    for col in ['TenureMonths', 'TotalUsageGB', 'ServiceType', 'PaymentMethod']:
        main_df[col] = rng.normal(0, 1, size=n).round(2)

    main_df['CustomerID'] = [f"ID{i:05d}" for i in range(n)]

    z = (base - base.mean()) / (base.std() + 1e-9)
    classes = ['이탈 여부', '이탈_여부']
    if '분류' == "분류":
        if False and len(classes) >= 3:
            bins = np.quantile(z, [1 / 3, 2 / 3])
            idx = np.digitize(z, bins)
            main_df['Churned'] = [classes[i] for i in idx]
        else:
            prob = 1 / (1 + np.exp(-z))
            labels = (rng.random(n) < prob).astype(int)
            main_df['Churned'] = np.where(labels == 1, classes[0], classes[-1])
    else:
        noise = rng.normal(0, 5, size=n)
        main_df['Churned'] = (base * 1.5 + noise).round(2)

    for col in ['MonthlyCharge'] + ['TenureMonths', 'TotalUsageGB', 'ServiceType', 'PaymentMethod'][:1]:
        na_idx = rng.choice(n, size=int(n * 0.03), replace=False)
        main_df.loc[na_idx, col] = np.nan

    main_df = main_df.sample(frac=1, random_state=seed).reset_index(drop=True)

    dim_df = pd.DataFrame({
        'RegionCode': regions,
        "dim_value": rng.uniform(0.5, 2.0, size=n_regions).round(3),
    })
    return main_df, dim_df


main_df, dim_df = synthesize_tables()
main_df.to_json("data.json", index=False)
dim_df.to_csv("dim.csv", index=False)
print("데이터 저장 완료 - data.json:", main_df.shape, "/ dim.csv:", dim_df.shape)
main_df.head(4)

## <데이터 분석>

### 1. 라이브러리 임포트

pandas, numpy, matplotlib.pyplot, seaborn 을 각각 pd, np, plt, sns 별칭으로 임포트하세요.

In [ ]:
# (1) 여기에 답안코드를 작성하고 실행하세요



### 2. 데이터 로드 (read_csv/read_json)

`data.json` 를 `pd.read_json` 로 읽어 **my_data** 에, `dim.csv` 를 `pd.read_csv` 로 읽어 **dim_data** 에 각각 할당하세요.

In [ ]:
# (2) 여기에 답안코드를 작성하고 실행하세요



### 3. 결측치 확인

my_data 의 `MonthlyCharge` 컬럼에 결측치(NaN)가 몇 개 있는지 `isna().sum()` 으로 확인하세요. 몇 개입니까?

In [ ]:
# (3) 정답을 answer_3 변수에 저장하세요 (실행하세요)

answer_3 = ""


### 4. 데이터 병합 (pd.merge)

`RegionCode` 를 키로 my_data 와 dim_data 를 **left join** 하여 **data_merged** 에 저장하세요.

In [ ]:
# (4) 여기에 답안코드를 작성하고 실행하세요



### 5. 데이터 집계 (groupby)

`ContractType` 별 `MonthlyCharge` 의 평균을 구해 **df_grp** 에 저장하세요.

In [ ]:
# (5) 여기에 답안코드를 작성하고 실행하세요



### 6. 시각화 (subplots)

계약 유형 (단기/장기/위약금 없음)(ContractType) 분포의 countplot 과, 이탈 여부, 이탈_여부 별 월별 청구 금액 (원)(MonthlyCharge) histplot 을 나란히 그리는 코드입니다.
빈칸 **(A)** 에 들어갈, 여러 그래프를 한 번에 그릴 때 쓰는 matplotlib 함수 이름은?

```python
fig, axes = plt.(A)(nrows=1, ncols=2, figsize=(12, 5))
sns.countplot(data=data_merged, x='ContractType', ax=axes[0])
sns.histplot(data=data_merged, x='MonthlyCharge', hue='Churned', ax=axes[1])
plt.show()
```

In [ ]:
# (6) 정답을 answer_6 변수에 저장하세요 (실행하세요)

answer_6 = ""


### 7. 시각화 2

이탈 여부, 이탈_여부 별 월별 청구 금액 (원)(MonthlyCharge) 분포를 seaborn boxplot 으로 시각화하세요.

In [ ]:
# (7) 여기에 답안코드를 작성하고 실행하세요



## <데이터 전처리>

### 8. 이상치 처리

IQR 기준(K=1.0)을 벗어나는 이상치 행을 제거하고, 식별자 컬럼도 삭제해서 **data_temp** 에 저장하는 코드입니다. 빈칸 **(A)** 에 들어갈, 행을 삭제할 때 쓰는 DataFrame 메서드 이름을 파악한 뒤, 아래 코드 전체를 (A)를 채워서 작성하고 실행하세요 (이후 문항들이 data_temp 를 사용합니다).

```python
q1 = data_merged['MonthlyCharge'].quantile(0.25)
q3 = data_merged['MonthlyCharge'].quantile(0.75)
iqr = q3 - q1
lower_fence = q1 - 1.0 * iqr
upper_fence = q3 + 1.0 * iqr
data_temp = data_merged.(A)(data_merged[(data_merged['MonthlyCharge'] > upper_fence) | (data_merged['MonthlyCharge'] < lower_fence)].index)
data_temp = data_temp.drop(columns=['CustomerID'])
data_temp = data_temp.reset_index(drop=True)
```

In [ ]:
# (8) 위 코드에서 (A)를 채운 전체 코드를 작성하고 실행하세요



### 9. 결측치 처리

다음은 data_temp 의 결측치를 대표값으로 채우는 코드인데, 실행하면 의도한 것과 다르게 동작합니다. 무엇이 문제인지 서술하거나, 올바르게 고친 코드를 작성하세요.

```python
fill_value = data_temp['MonthlyCharge'].mean()
data_na = data_temp.fillna({'MonthlyCharge': fill_value})
data_na['TenureMonths'] = data_na['TenureMonths'].fillna(data_na['TenureMonths'].mean())
# (이 버전에는 의도한 것과 다른 결과를 내는 부분이 있습니다)
```

In [ ]:
# (9) 여기에 문제점 설명 또는 수정한 코드를 작성하세요



### 10. 인코딩

`ContractType` 는 원-핫 인코딩(get_dummies, drop_first=True), `RegionCode` 는 sklearn LabelEncoder 로 인코딩해서 data_preset 에 저장하세요. 타깃 컬럼 `Churned` 도 LabelEncoder 로 정수 인코딩하세요.

In [ ]:
# (10) 여기에 답안코드를 작성하고 실행하세요



### 11. 데이터 분리

Churned 을 y, 나머지를 X 로 삼아 train_test_split 으로 분리하세요.
- test_size=0.25, random_state=7, stratify 옵션을 적용하세요
- 변수명: X_train, X_valid, y_train, y_valid

In [ ]:
# (11) 여기에 답안코드를 작성하고 실행하세요



### 12. 스케일링

RobustScaler 로 훈련/검증 데이터를 스케일링하는 코드인데, 의도한 것과 다르게 동작합니다. 무엇이 문제인지 서술하거나, 올바르게 고친 코드를 작성하세요.

```python
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()
X_valid_scaled = scaler.fit_transform(X_train)
X_train_scaled = scaler.transform(X_valid)
```

In [ ]:
# (12) 여기에 문제점 설명 또는 수정한 코드를 작성하세요



### 13. 스케일러 특성

RobustScaler 를 훈련 데이터에 적용하면, 결과 값의 분포는 이론적으로 어떤 특성을 가지게 됩니까?

In [ ]:
# (13) 정답을 answer_13 변수에 저장하세요 (실행하세요)

answer_13 = ""


## <AI 모델링>

### 14. 머신러닝 기본 (fit-predict)

DecisionTreeClassifier 로 모델을 하나 만들어 학습시키고, 검증데이터에 대한 예측값을 **pred_y** 에 저장하세요. (변수명: model, pred_y)

In [ ]:
# (14) 여기에 답안코드를 작성하고 실행하세요



### 15. GridSearch 모델링

DecisionTreeClassifier 와 XGBClassifier 를 GridSearchCV(cv=5, scoring='f1_macro')로 탐색하고 학습하세요.
- max_depth 후보: [3, 5, 7]
- XGBClassifier 의 n_estimators 후보: [50, 100, 200]
- 변수명: gs_a (베이스 모델), gs_b (비교 모델)

In [ ]:
# (15) 여기에 답안코드를 작성하고 실행하세요



### 16. GridSearch 결과 확인

위 GridSearch에서 XGBClassifier 의 n_estimators 후보는 [50, 100, 200] 였습니다. GridSearchCV가 고를 수 있는 값의 후보 중 '가장 큰 값'은 얼마인가요?

In [ ]:
# (16) 정답을 answer_16 변수에 저장하세요 (실행하세요)

answer_16 = ""


### 17. 변수중요도

XGBClassifier 의 변수중요도 Top 15개(정렬 ascending=False)를 뽑아 bar 차트로 시각화하는 코드인데, 의도한 것과 다르게 동작합니다. 무엇이 문제인지 서술하거나, 올바르게 고친 코드를 작성하세요.

```python
fi = pd.DataFrame({'feature': X_train.columns, 'importance': gs_b.best_estimator_.feature_importances_})
fi = fi.sort_values('importance', ascending=False)[lambda x: x > 0][:15]
sns.barplot(x='feature', y='importance', data=fi)
plt.show()
```

In [ ]:
# (17) 여기에 문제점 설명 또는 수정한 코드를 작성하세요



### 18. 성능평가

검증데이터로 gs_a, gs_b 두 모델의 **roc_auc_score** 를 각각 계산해서 a_score, b_score 에 저장하세요 (predict_proba 의 양성 클래스(1) 확률을 사용하세요. 타깃은 인코딩 단계에서 이미 0/1 정수로 변환되어 있습니다).

In [ ]:
# (18) 여기에 답안코드를 작성하고 실행하세요



### 19. 모델 성능 비교

바로 위에서 계산한 a_score 와 b_score 를 비교했을 때, 어느 모델이 더 우수하다고 판단할 수 있습니까? (둘 중 하나를 실행 결과에 따라 답하세요)

In [ ]:
# (19) 정답을 answer_19 변수에 저장하세요 (실행하세요)

answer_19 = ""


### 20. 딥러닝 설계

다음은 딥러닝 모델을 설계하는 코드입니다. EarlyStopping 에서 '몇 epoch 동안 개선이 없으면 멈출지' 지정하는 파라미터 이름(빈칸 **(A)**)을 파악한 뒤, 아래 코드 전체를 (A)를 채워서 작성하고 실행하세요 (이후 문항들이 model/cb_list 를 사용합니다). (은닉층 활성함수: relu, 출력층: sigmoid/binary_crossentropy, ModelCheckpoint 포함)

```python
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

model = Sequential([
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
cb_list = [EarlyStopping(monitor='val_loss', (A)=20, restore_best_weights=True)]
cb_list.append(ModelCheckpoint('best_model.keras', monitor='val_loss', save_best_only=True))
```

In [ ]:
# (20) 위 코드에서 (A)를 채운 전체 코드를 작성하고 실행하세요



### 21. 딥러닝 학습

위에서 설계한 model 을 batch_size=16, epochs=50 으로 학습하고 history 에 저장하세요 (callbacks=cb_list 사용).

In [ ]:
# (21) 여기에 답안코드를 작성하고 실행하세요



### 22. 학습곡선 시각화

history 를 이용해서 학습/검증 **accuracy** 변화를 한 그래프에 시각화하세요 (x축 라벨: epoch, 범례 위치: upper left, 범례 텍스트: train/val).

In [ ]:
# (22) 여기에 답안코드를 작성하고 실행하세요



### 23. 저장된 모델 재사용

ModelCheckpoint 로 저장된 `best_model.keras` 를 `load_model()` 로 다시 불러와서, 검증데이터에 대한 예측값을 **reload_pred** 에 저장하세요.

In [ ]:
# (23) 여기에 답안코드를 작성하고 실행하세요



---
## 해설

스스로 풀어본 뒤 아래에서 확인하세요. 문항 번호가 위 문제 번호와 일치합니다.

### 1번 해설 - 라이브러리 임포트 [코드작성]

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

> import ... as ... 문법으로 널리 쓰이는 관례적 별칭을 지정합니다.

### 2번 해설 - 데이터 로드 (read_csv/read_json) [코드작성]

In [ ]:
my_data = pd.read_json('data.json')
dim_data = pd.read_csv('dim.csv')
my_data.head(4)

> pd.read_json() 로 파일 형식에 맞는 로더를 사용합니다.

### 3번 해설 - 결측치 확인 [결과값예측]

In [ ]:
18

> Series.isna().sum() 은 True(결측치)의 개수를 셉니다.

### 4번 해설 - 데이터 병합 (pd.merge) [코드작성]

In [ ]:
data_merged = pd.merge(my_data, dim_data, on='RegionCode', how='left')
data_merged.head(4)

> pd.merge(left, right, on=키, how='left') 는 왼쪽 테이블 기준으로 오른쪽 테이블을 결합합니다.

### 5번 해설 - 데이터 집계 (groupby) [코드작성]

In [ ]:
df_grp = data_merged.groupby('ContractType')['MonthlyCharge'].mean()
df_grp

> groupby(기준컬럼)[대상컬럼].mean() 형태로 그룹별 집계를 구합니다.

### 6번 해설 - 시각화 (subplots) [빈칸채우기]

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12, 5))
sns.countplot(data=data_merged, x='ContractType', ax=axes[0])
sns.histplot(data=data_merged, x='MonthlyCharge', hue='Churned', ax=axes[1])
plt.show()

> plt.subplots() 는 nrows/ncols 로 여러 축(Axes)을 한 번에 만듭니다. 정답: subplots

### 7번 해설 - 시각화 2 [코드작성]

In [ ]:
sns.boxplot(data=data_merged, x='Churned', y='MonthlyCharge')
plt.show()

> sns.boxplot(x=, y=) / sns.jointplot(x=, y=) 형태로 그립니다.

### 8번 해설 - 이상치 처리 [빈칸채우기]

In [ ]:
q1 = data_merged['MonthlyCharge'].quantile(0.25)
q3 = data_merged['MonthlyCharge'].quantile(0.75)
iqr = q3 - q1
lower_fence = q1 - 1.0 * iqr
upper_fence = q3 + 1.0 * iqr
data_temp = data_merged.drop(data_merged[(data_merged['MonthlyCharge'] > upper_fence) | (data_merged['MonthlyCharge'] < lower_fence)].index)
data_temp = data_temp.drop(columns=['CustomerID'])
data_temp = data_temp.reset_index(drop=True)

> DataFrame.drop() 은 행(기본 axis=0) 또는 열(axis=1)을 삭제합니다. 정답: drop

### 9번 해설 - 결측치 처리 [오류정정]

In [ ]:
fill_value = data_temp['MonthlyCharge'].mean()
data_na = data_temp.fillna({'MonthlyCharge': fill_value})
data_na['TenureMonths'] = data_na['TenureMonths'].fillna(data_na['TenureMonths'].mean())

> [loop_control] 루프/조건 제어 착각: 조건문을 잘못 넣어 일부 로직이 건너뛰어지거나 잘못 실행됨

### 10번 해설 - 인코딩 [코드작성]

In [ ]:
data_preset = pd.get_dummies(data=data_na, columns=['ContractType'], drop_first=True)

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
data_preset['RegionCode'] = le.fit_transform(data_preset['RegionCode'])
data_preset['Churned'] = LabelEncoder().fit_transform(data_preset['Churned'])
data_preset.info()

> 저-카디널리티는 원-핫, 코드성 범주는 라벨 인코딩을 흔히 사용합니다. 타깃 컬럼도 XGBoost/LightGBM 등 일부 모델은 문자열 클래스를 그대로 받아들이지 않으므로, `Churned` 도 함께 LabelEncoder 로 0/1(다중클래스는 0..N-1) 정수로 인코딩합니다.

### 11번 해설 - 데이터 분리 [코드작성]

In [ ]:
from sklearn.model_selection import train_test_split

X = data_preset.drop(['Churned'], axis=1)
y = data_preset['Churned']

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.25, random_state=7, stratify=y)

> train_test_split(X, y, ...) 은 X_train, X_valid, y_train, y_valid 순서로 반환합니다.

### 12번 해설 - 스케일링 [오류정정]

In [ ]:
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

> [scope_reference] 이 코드는 훈련 데이터(X_train)를 사용하여 스케일러를 학습시킨 후, 검증 데이터(X_valid)를 변환하는 의도를 뒤바꾸었습니다. 결과적으로 검증 데이터에 대해 훈련 데이터의 파라미터를 적용하여 잘못된 스케일링 결과를 얻게 됩니다.

### 13번 해설 - 스케일러 특성 [결과값예측]

In [ ]:
'중앙값(median) 0 부근, IQR(사분위범위) 기준으로 스케일링되어 이상치 영향을 덜 받습니다.'

> RobustScaler 의 정의에 따른 이론적 특성입니다.

### 14번 해설 - 머신러닝 기본 (fit-predict) [코드작성]

In [ ]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(random_state=7)
model.fit(X_train_scaled, y_train)
pred_y = model.predict(X_valid_scaled)
pred_y[:5]

> import → model = 클래스() → model.fit(X_train, y_train) → pred_y = model.predict(X_valid) 4줄 템플릿입니다.

### 15번 해설 - GridSearch 모델링 [코드작성]

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV

gs_a = GridSearchCV(DecisionTreeClassifier(random_state=7), {'max_depth':[3,5,7]}, cv=5, scoring='f1_macro')
gs_a.fit(X_train_scaled, y_train)

gs_b = GridSearchCV(XGBClassifier(random_state=7), {'n_estimators':[50, 100, 200], 'max_depth':[3,5,7]}, cv=5, scoring='f1_macro')
gs_b.fit(X_train_scaled, y_train)

> GridSearchCV(estimator, param_grid, cv=...).fit(X_train, y_train) 형태로 탐색합니다. XGBClassifier 는 sklearn 기본 앙상블 외에 XGBoost/LightGBM 계열일 수도 있습니다.

### 16번 해설 - GridSearch 결과 확인 [결과값예측]

In [ ]:
200

> 제시된 후보 중 GridSearch가 고를 수 있는 최댓값을 묻는 문항입니다.

### 17번 해설 - 변수중요도 [오류정정]

In [ ]:
fi = pd.DataFrame({'feature': X_train.columns, 'importance': gs_b.best_estimator_.feature_importances_})
fi = fi.sort_values('importance', ascending=False)[:15]
sns.barplot(x='feature', y='importance', data=fi)
plt.show()

> [loop_control] 원래 코드는 단순히 상위 15개 특징을 선택하는 것이 목표였으나, 오류가 삽입된 버전에서는 `sort_values` 후 슬라이싱(`[:15]`) 전에 불필요하거나 잘못된 조건(`lambda x: x > 0`)을 추가했습니다. 이 조건은 데이터프레임을 필터링하는 과정에서 의도치 않게 일부 중요한 데이터를 건너뛰어(혹은 조건에 맞지 않는 행을 제거하여) 시각화 결과가 부정확해질 수 있습니다.

### 18번 해설 - 성능평가 [코드작성]

In [ ]:
from sklearn.metrics import roc_auc_score

proba_a = gs_a.best_estimator_.predict_proba(X_valid_scaled)[:, 1]
proba_b = gs_b.best_estimator_.predict_proba(X_valid_scaled)[:, 1]

a_score = roc_auc_score(y_valid, proba_a)
b_score = roc_auc_score(y_valid, proba_b)
print(a_score, b_score)

> roc_auc_score(정답, 예측확률) 형태이며 predict()가 아닌 predict_proba() 의 확률값을 사용합니다. 타깃이 이미 0/1 정수이므로 별도 이진화가 필요 없습니다.

### 19번 해설 - 모델 성능 비교 [결과값예측]

In [ ]:
'실행 결과에 따라 달라집니다: 값이 더 큰 쪽 이 더 우수한 모델입니다. a_score, b_score 를 직접 비교해서 판단하세요.'

> 정확도/F1/ROC-AUC 등은 높을수록, MAE/MSE 등 오차 지표는 낮을수록 좋은 성능입니다.

### 20번 해설 - 딥러닝 설계 [빈칸채우기]

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

model = Sequential([
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
cb_list = [EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)]
cb_list.append(ModelCheckpoint('best_model.keras', monitor='val_loss', save_best_only=True))

> EarlyStopping(patience=N) 은 N번 연속 개선이 없으면 학습을 멈춥니다. 정답: patience

### 21번 해설 - 딥러닝 학습 [코드작성]

In [ ]:
history = model.fit(X_train_scaled, y_train, epochs=50, batch_size=16,
                    validation_data=(X_valid_scaled, y_valid), callbacks=cb_list)

> model.fit(X, y, epochs=, batch_size=, validation_data=, callbacks=) 형태입니다.

### 22번 해설 - 학습곡선 시각화 [코드작성]

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model accuracy')
plt.xlabel('epoch')
plt.ylabel('accuracy')
plt.legend(['train', 'val'], loc='upper left')
plt.show()

> history.history[지표] 로 epoch별 기록을 꺼내 plt.plot() + plt.legend(loc=...) 으로 그립니다.

### 23번 해설 - 저장된 모델 재사용 [코드작성]

In [ ]:
from tensorflow.keras.models import load_model

saved_model = load_model('best_model.keras')
reload_pred = saved_model.predict(X_valid_scaled)
reload_pred[:5]

> 학습 없이도 저장된 가중치를 불러와(load_model) 바로 predict() 할 수 있습니다.